<a href="https://colab.research.google.com/github/RaselParvejHrid/machine-learning/blob/main/with-chatgpt/001%20KNN/project002/project002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metadata

[Project Details](https://github.com/RaselParvejHrid/machine-learning/tree/main/with-chatgpt/001%20KNN/project005)

Started on Jul 08, 2026

Completed on Jul 08, 2026

# Lib Imports

In [ ]:

import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from IPython.display import display
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Loading the Dataset as SKLearn#Bunch

In [ ]:
print(load_wine().DESCR)

This claims that there is no attribute value missing. The number of samples is 178. There are 13 predictive numeric features, and a target class with 3 values in its domain.

Still I will check these again with Pandas#Dataframe API.

Also, the feature values have variying min-max range. And as my algorithm here will be KNN that is distance-based, scaling of features is must.



# Loading the Dataset as Pandas#Dataframe

In [ ]:
dataset = load_wine(as_frame=True).frame

In [ ]:
dataset

# EDA

## Analytical View of the Dataset

### Dataset Shape

In [ ]:
n_rows, n_columns = dataset.shape

In [ ]:
print(f"The dataset has\n{n_rows} samples\nand {n_columns} columns.")

Out of 14 columns, 'target' is a categorical target and, other 13 columns are numeric features.

### Dataset Info

In [ ]:
dataset.info()

### Statistical Profiling

In [ ]:
dataset.describe()

Features has varying min-max ranges. Yes, scaling of features is a must.

### Samples

In [ ]:
dataset.head(10)

In [ ]:
dataset.tail(10)

In [ ]:
dataset.sample(10)

## Feature Screening

### Correctness of Data Type

In [ ]:
dataset.info()

Correct types.

### Constant Feature

In [ ]:
dataset.sample(10)

No constant Feature

### Irrelevant Feature

In [ ]:
dataset.info()

I do not have the domain knowledge here. Just from the names, no feature sounds irrelevant.

### Missing Values

In [ ]:
dataset.isnull().sum()

No null values.

### Duplicated Rows

In [ ]:
dataset.duplicated().sum()

## Univariate Analysis

14 columns. The 'target' column is categorical and is the label for our model to predict.

And other 13 columns are numeric features. No categorical feature.

In [ ]:
def get_frequency_table(col: pd.Series, sort: bool = True):
    freq_table = col.value_counts(sort=sort).to_frame(name='Frequencies')
    freq_table['Cumulative Frequencies'] = freq_table['Frequencies'].cumsum()
    freq_table['Percentages'] = (col.value_counts(normalize=True, sort=False) * 100)
    freq_table['Cumulative Percentages'] = freq_table['Percentages'].cumsum()
    return freq_table

#### Categorical Features

In [ ]:
def cat_column_analysis(column: pd.Series, column_name: str = None):

    if not isinstance(column, pd.Series):
        raise TypeError("The column argument must be a pandas Series.")
        
    if not isinstance(column.dtype, pd.CategoricalDtype):
        raise TypeError("The provided Series must have a categorical dtype (and a .cat accessor).")
    
    if column_name == None:
        column_name = column.name

    # Frequency Table
    freq_table = get_frequency_table(column, sort=not column.cat.ordered)
    print(f"Frequency Table for the Column '{column_name}'", freq_table, sep='\n\n')

    # Central Tendencies
    print(f"\n\nCentral Tendencies of the Column '{column_name}'\n")
    print(f"Mode: '{column.mode()[0]}'")

    if column.cat.ordered:
        sorted_column = column.sort_values()
        column_length = column.shape[0]
        if column_length % 2 == 1:
            print(f"Median: '{sorted_column[(column_length-1) // 2]}'")
        else:
            median1index = column_length // 2
            median2index = median1index + 1
            print(f"Median: '{sorted_column[median1index]}' and '{sorted_column[median2index]}'")

    # Plot
    countplot = sns.catplot(
        data=pd.DataFrame(column),
        x="target",
        kind= "count",
        order= column.cat.categories
    )

    countplot.set_titles(f"Count Plot for the Column '{column_name}'")
    countplot.set_axis_labels(column_name, 'Frequency')
    countplot.set_xticklabels(rotation=90, horizontalalignment='right')
    

#### Numeric Features

In [ ]:
def num_column_histplot(column: pd.Series, column_name: str, bins: int = 50):
    fig = Figure(figsize=(6, 4))
    ax = fig.add_subplot(111)

    sns.histplot(x=column, bins=bins, ax=ax)
    ax.set_xlabel(column_name)

    fig.canvas.draw()
    display(fig)


def num_column_kdeplot(column: pd.Series, column_name: str):
    mean = column.mean()
    std = column.std()
    z_scores = np.array([-3, -2, -1, 0, 1, 2, 3])
    tick_positions = mean + (z_scores * std)
    tick_labels = [
        f"{z:+d}σ ({pos:.1f})" if z != 0 else f"μ ({pos:.1f})"
        for z, pos in zip(z_scores, tick_positions)
    ]

    fig = Figure(figsize=(6, 4))
    ax = fig.add_subplot(111)

    sns.kdeplot(x=column, ax=ax)
    ax.grid(True, which="both", axis="both")

    secx = ax.secondary_xaxis("top")
    secx.set_xticks(tick_positions)
    secx.set_xticklabels(tick_labels, rotation=90)
    secx.set_xlabel("z-score")
    secx.grid(True, color="gray", linestyle="--", alpha=0.6)

    for z, pos in zip(z_scores, tick_positions):
        if z == 0:
            ax.axvline(
                pos, color="red", linestyle="-", alpha=0.5, linewidth=1.2
            )
        else:
            ax.axvline(
                pos, color="red", linestyle="--", alpha=0.5, linewidth=1
            )

    ax.set_xlabel(column_name)


    fig.canvas.draw()
    display(fig)


def num_column_boxplot(column: pd.Series, column_name: str):
    fig = Figure(figsize=(6, 2))
    ax = fig.add_subplot(111)

    sns.boxplot(x=column, ax=ax)
    ax.set_xlabel(column_name)

    fig.canvas.draw()
    display(fig)

In [ ]:
def num_column_analysis(column: pd.Series, column_name: str = None):

    if not isinstance(column, pd.Series):
        raise TypeError("The column argument must be a pandas Series.")
        
    if not pd.api.types.is_numeric_dtype(column):
        raise TypeError("The provided Series must be a numerical dtype.")
    
    if column_name == None:
        column_name = column.name

    # Frequency Table
    freq_table = get_frequency_table(column, sort=True)
    print(f"Frequency Table for the Column '{column_name}'", freq_table, sep='\n\n')

    # Central Tendencies
    print(f"\n\nCentral Tendencies of the Column '{column_name}'\n")
    print(f"Mean: {column.mean()}")
    print(f"Median: {column.median()}")
    print(f"Mode: {column.mode()[0]}")

    # Plots
    print(f"\n\nKDE Plot for the Column '{column_name}'\n")
    num_column_kdeplot(column, column_name)

    # print(f"\n\nBox Plot for the Column '{column_name}'\n")
    # num_column_boxplot(column, column_name)

    # print(f"\n\nHistogram for the Column '{column_name}'\n")
    # num_column_histplot(column, column_name)
    

##### alcohol

In [ ]:
num_column_analysis(dataset['alcohol'], 'Alcohol')

##### malic_acid

In [ ]:
num_column_analysis(dataset['malic_acid'], 'Malic acid')

##### ash

In [ ]:
num_column_analysis(dataset['ash'], 'Ash')

##### alcalinity_of_ash

In [ ]:
num_column_analysis(dataset['alcalinity_of_ash'], 'Alcalinity of ash')

##### magnesium

In [ ]:
num_column_analysis(dataset['magnesium'], 'Magnesium')

##### total_phenols

In [ ]:
num_column_analysis(dataset['total_phenols'], 'Total phenols')

##### flavanoids

In [ ]:
num_column_analysis(dataset['flavanoids'], 'Flavanoids')

##### nonflavanoid_phenols

In [ ]:
num_column_analysis(dataset['nonflavanoid_phenols'], 'Nonflavanoid phenols')

##### proanthocyanins

In [ ]:
num_column_analysis(dataset['proanthocyanins'], 'Proanthocyanins')

##### color_intensity

In [ ]:
num_column_analysis(dataset['color_intensity'], 'Color intensity')

##### hue

In [ ]:
num_column_analysis(dataset['hue'], 'Hue')

##### od280/od315_of_diluted_wines

In [ ]:
num_column_analysis(dataset['od280/od315_of_diluted_wines'], 'OD280/OD315 of diluted wines')

##### proline

In [ ]:
num_column_analysis(dataset['proline'], 'Proline')

### The Label

In [ ]:
label_domain = ['Class 0', 'Class 1', 'Class 2']
label = dataset['target'].astype(pd.CategoricalDtype(categories=[0, 1, 2], ordered=True)).cat.rename_categories(label_domain)

cat_column_analysis(label, 'Target (Wine Class)')

# Feature-Target Separation

In [ ]:
X = dataset.drop('target', axis=1)
y = dataset['target']

# Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Pipeline

In [ ]:
knn_pipeline = Pipeline([
    ('standard_scaling', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# HyperParameter Tuning

In [ ]:
grid_params = {
    "knn__n_neighbors": range(1, 16, 2),
    "knn__weights": ['uniform', 'distance'],
    'knn__p' : [1, 2]
}

grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=grid_params,
    cv=5
)

In [ ]:
grid.fit(X_train, y_train)

# Best Model

In [ ]:
print(f"The best Value of k is {grid.best_params_['knn__n_neighbors']}")

In [ ]:
print(f"The best weight is {grid.best_params_['knn__weights']}")

In [ ]:
print(f"The best distance metric is {['Euclidean', 'Manhattan'][grid.best_params_['knn__p']]}")

In [ ]:
best_knn_model = grid.best_estimator_

# Test

In [ ]:
y_test_pred = best_knn_model.predict(X_test)

In [ ]:
print(accuracy_score(y_test, y_test_pred))

In [ ]:
my_confusion_matrix = confusion_matrix(y_test, y_test_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(my_confusion_matrix, display_labels=label_domain)

cm_display.plot()



In [ ]:
print(classification_report(y_test, y_test_pred, target_names=label_domain))